# SFT (Supervised Fine-Tuning)

- **목적**: 난독화된 한국어 리뷰를 원문으로 복원하는 모델 학습
- **베이스 모델**: `AIDX-ktds/ktdsbaseLM-v0.13-onbased-llama3.1` (Llama 3.1 기반 한국어 모델)
- **학습 방식**: QLoRA (4bit 양자화 + LoRA 어댑터)
- **프롬프트 포맷**: 난독화 복원 전용 포맷 형식 (`### Instruction` / `### Response`)
- **데이터**: 대회 제공 `train.csv` + 난독화 사이트 증강 데이터 결합 (총 약 34,000개)
- **학습 steps**: 1,100 steps

In [ ]:
!pip install accelerate==1.2.1 bitsandbytes==0.45.1 peft==0.14.0 transformers==4.48.2 trl==0.14.0
!pip install flash-attn --no-build-isolation
!pip install vllm==0.7.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

# ── 경로 설정 ─────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/Colab Notebooks/open'
CKPT_DIR    = '/content/drive/MyDrive/Colab Notebooks/model_checkpoint/fine_tune_output_llama_3.1'

TRAIN_CSV   = os.path.join(BASE_DIR, 'train.csv')
AUG_CSV     = os.path.join(BASE_DIR, 'augmented_texts.csv')
AUG_CSV_50  = os.path.join(BASE_DIR, 'augmented_texts50.csv')
AUG_CSV_100 = os.path.join(BASE_DIR, 'augmented_texts100.csv')

os.makedirs(CKPT_DIR, exist_ok=True)

## 1. 데이터 로드 및 전처리

- 대회 제공 `train.csv` + 난독화 사이트에서 증강한 데이터 3종 결합
- 중복 제거 후 학습에 사용

In [ ]:
train      = pd.read_csv(TRAIN_CSV,   encoding='utf-8-sig')
aug_texts  = pd.read_csv(AUG_CSV,     encoding='utf-8-sig')
aug_50     = pd.read_csv(AUG_CSV_50,  encoding='utf-8-sig')
aug_100    = pd.read_csv(AUG_CSV_100, encoding='utf-8-sig')

# 데이터 결합 + 중복 제거
train_combined = pd.concat([train, aug_texts, aug_50, aug_100], ignore_index=True)
train_combined = train_combined.drop_duplicates().reset_index(drop=True)

print(f'Train 데이터 크기: {len(train_combined)}')
train_combined.head(3)

## 2. 모델 로드 (QLoRA)

- **Quantization**: 4bit NF4 양자화로 VRAM 절약
- **LoRA**: r=16, lora_alpha=32, Attention + FFN 전체 레이어 대상
  - 데이터셋 33,000개 / 중간 크기 모델(7~13B) 기준 r=16, lora_alpha=32 적용
- **Flash Attention 2**: 긴 시퀀스 처리 속도 및 메모리 효율 향상

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
import torch

MODEL_NAME = 'AIDX-ktds/ktdsbaseLM-v0.13-onbased-llama3.1'

# 4bit 양자화 설정
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

# LoRA 설정
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    attn_implementation='flash_attention_2',
    device_map={'': 0}
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({'pad_token': '<|reserved_special_token_247|>'})
model.config.pad_token_id = tokenizer.pad_token_id

## 3. 프롬프트 포맷 설정

- 난독화 복원 전용 포맷: 시스템 프롬프트 + `### Instruction` + `### Response`
- `DataCollatorForCompletionOnlyLM`으로 `### Response:` 이후 부분만 loss 계산 대상으로 마스킹

In [ ]:
EOS_TOKEN = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
    "Your task is to transform the given obfuscated Korean review into a clear, correct, "
    "and natural-sounding Korean review that reflects its original meaning. "
    "Spacing and word length in the output must be restored to the same as in the input. "
    "Do not provide any description. Print only in Korean."
)

def build_deobfuscation_prompt(instruction, response):
    return (
        f"{SYSTEM_PROMPT}\n"
        f"\n\n### Instruction:\n{instruction}"
        f"\n\n### Response:\n{response}    "
    )

def formatting_prompts_func_for_sft(examples):
    texts = [
        build_deobfuscation_prompt(inst, out) + EOS_TOKEN
        for inst, out in zip(examples['input'], examples['output'])
    ]
    return {'text': texts}

In [ ]:
from datasets import Dataset
from trl import DataCollatorForCompletionOnlyLM

dataset = Dataset.from_pandas(train_combined)
dataset = dataset.shuffle(seed=42)

mapped_dataset = dataset.map(formatting_prompts_func_for_sft, batched=True)
split_dataset  = mapped_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset['train']
test_dataset  = split_dataset['test']

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')
print(train_dataset['text'][0])

In [ ]:
# ### Response: 이후 부분만 loss 계산 대상으로 남김
collator = DataCollatorForCompletionOnlyLM(
    response_template='### Response:\n',
    tokenizer=tokenizer,
    mlm=False
)

## 4. 학습 설정 및 실행

- 최종 증강 데이터 기준으로 처음부터 1,100 steps 학습
- `constant_with_warmup`: warmup_steps 이후 학습률 고정
- validation loss 모니터링을 위해 test_size=0.05 (약 1,700개) 설정

In [ ]:
%load_ext tensorboard
%tensorboard --logdir '{CKPT_DIR}/runs'

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=CKPT_DIR,
    report_to='tensorboard',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,  # 유효 배치 = 2 * 16 = 32
    warmup_steps=50,
    max_steps=1100,
    eval_steps=25,
    save_steps=50,
    evaluation_strategy='steps',
    save_strategy='steps',
    learning_rate=1e-4,
    logging_steps=1,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='constant_with_warmup',
    seed=42,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=training_arguments,
    dataset_text_field='text',
    data_collator=collator
)

In [ ]:
trainer.train()

## 5. 모델 병합 및 HuggingFace 업로드

- QLoRA 학습 후 LoRA 어댑터를 base 모델에 병합 (`merge_and_unload`)
- 병합된 full 모델을 HuggingFace에 업로드 → vLLM 추론 가능 상태
- 이후 DPO 데이터셋 구축 시 추론 모델로 재사용

In [ ]:
from peft import PeftModel

MERGED_MODEL_ID = 'Gyu96/llama3.1_additional_checkpoint1100'
FINAL_CKPT_PATH = os.path.join(CKPT_DIR, 'checkpoint-1100')

# 1. base 모델을 bfloat16으로 재로드 (양자화 없이, merge 용도)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={'': 0}
)

# 2. LoRA 어댑터 병합 후 어댑터 제거
base_model   = PeftModel.from_pretrained(base_model, FINAL_CKPT_PATH)
merged_model = base_model.merge_and_unload()

# 3. 병합된 full 모델 업로드
merged_model.push_to_hub(MERGED_MODEL_ID, token=True)
tokenizer.push_to_hub(MERGED_MODEL_ID, token=True)

## 6. SFT 모델 추론

- 업로드된 SFT 체크포인트(`checkpoint1100`)를 vLLM으로 로드해 테스트 데이터 추론
- `max_tokens`는 입력 문장 길이 기준으로 동적 설정 → 불필요한 생성 방지
- 추론 결과를 `SFT_inference_results.csv`로 저장

In [ ]:
import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams

SFT_INFER_MODEL = 'Gyu96/llama3.1_additional_checkpoint1100'
SFT_OUTPUT_CSV  = os.path.join(BASE_DIR, 'SFT_inference_results.csv')

SFT_PROMPT = (
    "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
    "Your task is to transform the given obfuscated Korean review into a clear, correct, "
    "and natural-sounding Korean review that reflects its original meaning. "
    "Spacing and word length in the output must be restored to the same as in the input. "
    "Do not provide any description. Print only in Korean."
    "\n\n### Instruction:\n{}\n\n### Response:\n{}"
)

vllm_sft = LLM(
    model=SFT_INFER_MODEL,
    tokenizer=SFT_INFER_MODEL,
    gpu_memory_utilization=0.9,
    max_model_len=8192
)

In [ ]:
def batch_inference(df, vllm_model, prompt_template, batch_size=128):
    """배치 단위로 추론을 실행하고 결과 리스트를 반환한다.
    입력 문장 길이 기준으로 max_tokens를 동적 설정해 불필요한 생성을 방지.
    """
    results = []

    for i in tqdm(range(0, len(df), batch_size), desc='Inference'):
        batch = df['input'].iloc[i:i + batch_size].tolist()

        max_tokens = max(len(text) for text in batch)
        sampling_params = SamplingParams(
            temperature=0.05,
            top_p=0.95,
            max_tokens=max_tokens
        )

        prompts = [prompt_template.format(text, '') for text in batch]
        outputs = vllm_model.generate(prompts, sampling_params)
        results.extend([o.outputs[0].text.strip() for o in outputs])

    return results

In [ ]:
# test_dataset → pandas DataFrame 변환
test_df = split_dataset['test'].to_pandas()

sft_preds = batch_inference(test_df, vllm_sft, SFT_PROMPT)

test_df['inference'] = sft_preds
test_df.to_csv(SFT_OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'SFT 추론 완료 → {SFT_OUTPUT_CSV}')
test_df[['input', 'output', 'inference']].head(5)